# Your 58-study validation set is lying to you — measured, with a fix

I trained four models for this competition and picked checkpoints on the 58 radiologist-annotated studies,
because they are the only real labels we have. Then the leaderboard told me the model my validation ranked
**last** was actually my **best**:

| model | 58-study val | 250-study derived holdout | public LB |
|---|---|---|---|
| v2 — single sagittal series, ResNet-34 | 0.858 | — | 0.873 |
| v3 — three planes + per-label plane mixture | **0.869** (best) | — | 0.874 |
| v4 — ConvNeXt-T + slice attention, EMA | 0.850 (worst) | **0.861** | **0.883** (best) |

The 58-study set ranked v4 last by 0.019. The leaderboard ranked it first by 0.009. The larger holdout — the
one built from *derived* labels, which are individually noisier but far more numerous — got the order right.

This notebook measures why that happens, then shows the fix. It is short and CPU-only.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

PATH = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])
train = pd.read_csv(os.path.join(PATH, 'train.csv'))
LABELS = list(train.columns[2:14])
ann = train[train[LABELS].notna().all(axis=1)].reset_index(drop=True)
print(f'annotated studies: {len(ann)} of {len(train)}')
print((ann[LABELS].sum().astype(int).rename('positives')).to_frame().assign(
      negatives=lambda d: len(ann) - d.positives).to_string())

## 1. How much can an AUC move on 58 studies, by luck alone?

Nine to thirty-five positives per label. To see what that means, take a **fixed** set of predictions — here,
a public LLM label table used as a stand-in for a model — and bootstrap the 58 studies. The predictions never
change; only which studies land in the sample does. Every bit of spread below is noise.

In [ ]:
src = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
tbl = pd.read_csv(src[0]).set_index('StudyInstanceUID')
pred = tbl.loc[ann['StudyInstanceUID'], LABELS].values.astype(float)
truth = ann[LABELS].values.astype(int)

rng = np.random.RandomState(0)
B = 2000
boot = np.zeros((B, len(LABELS))); boot_macro = np.zeros(B)
for b in range(B):
    idx = rng.randint(0, len(ann), len(ann))
    per = []
    for j in range(len(LABELS)):
        y, p = truth[idx, j], pred[idx, j]
        per.append(roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan)
    boot[b] = per
    boot_macro[b] = np.nanmean(per)

lo, hi = np.percentile(boot_macro, [2.5, 97.5])
print(f'macro AUC point estimate : {np.nanmean([roc_auc_score(truth[:, j], pred[:, j]) for j in range(12)]):.4f}')
print(f'95% bootstrap interval   : {lo:.4f} - {hi:.4f}   (width {hi-lo:.4f})')

**That interval width is the headline.** It is comparable to — often larger than — the entire spread between
my four models. Two models whose 58-study scores differ by less than that are indistinguishable, and picking
the higher one is a coin flip dressed up as a decision.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(boot_macro, bins=50, color='#4477AA')
axes[0].axvline(lo, color='#CC3311', ls='--'); axes[0].axvline(hi, color='#CC3311', ls='--')
axes[0].set_title(f'macro AUC over 2000 bootstraps of the same 58 studies\n95% CI width = {hi-lo:.3f}')
axes[0].set_xlabel('macro AUC')

w = np.nanpercentile(boot, 97.5, axis=0) - np.nanpercentile(boot, 2.5, axis=0)
o = np.argsort(-w)
axes[1].barh([LABELS[i] for i in o], w[o], color='#EE6677')
axes[1].set_xlabel('95% CI width (AUC)')
axes[1].set_title('Per-label uncertainty on 58 studies')
axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

print('per-label 95% CI width:')
for i in o: print(f'   {LABELS[i]:18s} +/- {w[i]/2:.3f}   ({int(truth[:, i].sum())} positives)')

Labels with the fewest positives (MCL at 9, Baker's at 12) carry CI widths around ±0.1. A model that "improves
MCL by 0.05" has told you nothing.

## 2. The fix: a large holdout with imperfect labels beats a small one with perfect labels

The instinct is that real radiologist labels must be the better yardstick. For *selection*, they are not — the
variance from n=58 swamps the bias from imperfect targets. Derived labels agree with radiologists at ~0.89
(see my [label benchmark](https://www.kaggle.com/code/starkhushi/rsna-knee-which-report-labels-should-you-train-on)),
and you can have hundreds of studies instead of dozens.

Simulate it: draw holdouts of increasing size from the derived-label pool and watch the interval collapse.

In [ ]:
pool_ids = train.loc[~train['StudyInstanceUID'].isin(ann['StudyInstanceUID']), 'StudyInstanceUID']
pool_ids = [u for u in pool_ids if u in tbl.index]
pool = tbl.loc[pool_ids, LABELS].apply(pd.to_numeric, errors='coerce').dropna()
print(f'derived-label pool available: {len(pool)} studies')

sizes = [58, 100, 250, 500, 1000]
widths = []
for n in sizes:
    m = []
    for b in range(300):
        idx = rng.choice(len(pool), n, replace=True)
        sub = pool.values[idx]
        y = (sub >= 0.5).astype(int)
        p = sub + rng.normal(0, 0.25, sub.shape)          # a model that is good but not perfect
        per = [roc_auc_score(y[:, j], p[:, j]) for j in range(12) if 0 < y[:, j].sum() < n]
        m.append(np.mean(per))
    widths.append(np.percentile(m, 97.5) - np.percentile(m, 2.5))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, widths, 'o-', color='#4477AA')
for x, y_ in zip(sizes, widths): ax.annotate(f'{y_:.3f}', (x, y_), textcoords='offset points', xytext=(0, 8))
ax.set_xlabel('holdout size (studies)'); ax.set_ylabel('95% CI width of macro AUC')
ax.set_title('Selection noise falls with holdout size, not with label purity')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3. Building that holdout without leaking

Two traps specific to this dataset:

**Duplicate reports.** 131 studies share a byte-identical report with another study (46 groups, the largest
spanning 37 studies). Since the targets are derived *from* the report, splitting such a group across
train/holdout scores the model on a target it was trained on. Keep groups whole.

**The annotated 58.** Keep them out of training entirely and report on them — just do not *select* on them.

In [ ]:
dup_groups = train.groupby('Report').size()
print(f'reports shared by >1 study: {(dup_groups > 1).sum()} groups, '
      f'{int(dup_groups[dup_groups > 1].sum())} studies, largest {dup_groups.max()}')

def make_split(train_df, label_tbl, n_holdout=250, seed=7):
    """Annotated studies -> report-only. Holdout drawn from studies with a UNIQUE report."""
    ann_ids = set(train_df[train_df[LABELS].notna().all(axis=1)]['StudyInstanceUID'])
    ann_reports = set(train_df[train_df['StudyInstanceUID'].isin(ann_ids)]['Report'])
    df = train_df[train_df['StudyInstanceUID'].isin(label_tbl.index)].copy()
    df = df[~(df['Report'].isin(ann_reports) & ~df['StudyInstanceUID'].isin(ann_ids))]   # report leak
    unique_report = ~df['Report'].duplicated(keep=False)
    cand = df[unique_report & ~df['StudyInstanceUID'].isin(ann_ids)]
    hold = set(cand.sample(n_holdout, random_state=seed)['StudyInstanceUID'])
    tr = df[~df['StudyInstanceUID'].isin(hold | ann_ids)]
    return tr['StudyInstanceUID'].tolist(), sorted(hold), sorted(ann_ids)

tr_ids, ho_ids, va_ids = make_split(train, tbl)
print(f'train {len(tr_ids)} | selection holdout {len(ho_ids)} | annotated report-only {len(va_ids)}')
assert not (set(tr_ids) & set(ho_ids)) and not (set(tr_ids) & set(va_ids))
print('no overlap between splits')

## What I changed, and what happened

From v5 onward I select checkpoints on the 250-study derived holdout and treat the 58 annotated studies as a
report-only sanity check. On the run where I first did this, the holdout curve rose monotonically while the
58-study number bounced ±0.02 between epochs — the two disagreed about the best epoch three times out of eight.

**Rules of thumb I would now follow in any competition with a tiny labelled set:**

- Bootstrap your validation metric once, at the start. If the CI is wider than the differences you intend to
  chase, that set cannot referee your experiments.
- Prefer more samples over cleaner labels *for selection*. Keep the clean set for a final, honest report.
- When a big holdout and a small clean set disagree, the big one is usually right about ranking.
- The leaderboard is a validation set too — with hundreds of studies behind it. Sometimes the cheapest way to
  break a tie between two models is one submission.

Corrections and counter-examples welcome. If this saved you from a bad model choice, an upvote helps others
find it.